# 05 - Features + Train/Test Split (Day 10)

Builds the feature matrix for the statistical model and makes the stratified split.

Features (one `FeatureUnion`): **word TF-IDF** (1-2 grams) + **char TF-IDF** (3-5 char-grams,
robust to romanisation spelling) + **lexicon counts** (slur terms per post, from Day 9's CSV).
The split is fit on **train only** to avoid leakage.

### Setup

In [4]:
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Not on Colab / already mounted:', e)

Not on Colab / already mounted: No module named 'google.colab'


In [5]:
import sys; sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')
from hinglish_hate import (load_bohra, load_hasoc2021, load_hasoc2022_threads,
                           build_corpus, filter_romanised,
                           load_lexicon, build_feature_union)
from sklearn.model_selection import train_test_split
import pandas as pd, joblib
from pathlib import Path
DATA_ROOT = Path(r'C:\Users\LENOVO\Desktop\dissertation\data')
def find_one(root,name):
    h=list(Path(root).rglob(name))
    if not h: raise FileNotFoundError(name)
    return h[0]
bohra=load_bohra(find_one(DATA_ROOT,'hate_speech.tsv'))
h21l=list(DATA_ROOT.rglob('labels.json'))
h21=load_hasoc2021(h21l[0].parents[2]) if h21l else None
h22=load_hasoc2022_threads(DATA_ROOT)
bohra_r=filter_romanised(bohra)
print('romanised Bohra (primary):', len(bohra_r))

[load_bohra] 4574 rows | fixed 0 label typo(s) | dropped 4 unparseable
romanised Bohra (primary): 4574


### 1. Load the curated lexicon

Reads only `keep==1` rows from Day 9's CSV (plus their variants). If you have not curated yet,
this will just be the seed terms - the pipeline still works and improves as you curate.

In [6]:
lex_csv = DATA_ROOT / 'hinglish_slur_lexicon.csv'
if lex_csv.exists():
    terms = load_lexicon(lex_csv)
else:
    terms = load_lexicon(find_one(DATA_ROOT, 'hate_lexicon.txt'))  # fallback to seed
    print('curated CSV not found - using seed hate_lexicon.txt')
print(f'{len(terms)} lexicon surface forms loaded')

curated CSV not found - using seed hate_lexicon.txt
171 lexicon surface forms loaded


## 2. Stratified train/test split

80/20 on romanised Bohra (the primary within-dataset corpus), stratified so the hate/not balance
is identical in both halves. HASOC stays whole as the cross-dataset test set (transformed with the
same union in `02`).

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    bohra_r['text'].tolist(), bohra_r['label'].tolist(),
    test_size=0.2, stratify=bohra_r['label'], random_state=42)
print(f'train {len(X_train)} (hate {sum(y_train)/len(y_train):.3f}) | '
      f'test {len(X_test)} (hate {sum(y_test)/len(y_test):.3f})')

train 3659 (hate 0.363) | test 915 (hate 0.363)


## 3. Build the feature matrix (fit on train only)

In [8]:
fu = build_feature_union(terms)
Xtr = fu.fit_transform(X_train)
Xte = fu.transform(X_test)

word = fu.transformer_list[0][1].vocabulary_
char = fu.transformer_list[1][1].vocabulary_
print(f'feature matrix: train {Xtr.shape}  test {Xte.shape}')
print(f'  word n-gram features : {len(word)}')
print(f'  char n-gram features : {len(char)}')
print(f'  lexicon features     : 2 (count, ratio)')

feature matrix: train (3659, 42039)  test (915, 42039)
  word n-gram features : 10355
  char n-gram features : 31682
  lexicon features     : 2 (count, ratio)


## 4. Save the split + fitted feature pipeline

So every later model trains and scores on the *identical* split and features.

In [9]:
split = DATA_ROOT / 'splits'
split.mkdir(exist_ok=True)
pd.DataFrame({'text':X_train,'label':y_train}).to_parquet(split/'bohra_train.parquet')
pd.DataFrame({'text':X_test, 'label':y_test }).to_parquet(split/'bohra_test.parquet')
joblib.dump(fu, split/'feature_union.joblib')
print('saved:')
for f in sorted(split.iterdir()): print('  ', f.name)

saved:
   bohra_test.parquet
   bohra_train.parquet
   feature_union.joblib


### Notes
- Char n-grams (`char_wb`, 3-5) are the main defence against romanisation spelling variation -
  they capture sub-word shape even when the exact word form is unseen.
- The lexicon branch is scaled (`MaxAbsScaler`) so its small counts do not get swamped by, or
  swamp, the TF-IDF features.
- Reload later with `joblib.load('splits/feature_union.joblib')` - do **not** refit on test data.
- To compare against the Day-1..10 baseline, drop the lexicon/char branches by rebuilding the union
  with only the word vectoriser; that isolates how much the new features add.